# 07b — Fetch Historical Weather for Duration Model

Fetches hourly weather data from [Open-Meteo Archive API](https://open-meteo.com/en/docs/historical-weather-api) for the dates in our Silver table (Jan 2015, Jan–Mar 2016) and saves it as a Delta table for the agent model to join during training.

**Run once** — the Delta table is then read by `07_trip_duration_agent.ipynb`.

In [0]:
%pip install requests --quiet

In [0]:
%restart_python

In [0]:
import requests
import pandas as pd
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

from src.constants import NYC_WEATHER_LAT, NYC_WEATHER_LON, WEATHER_TABLE

# Date ranges matching our Silver table
DATE_RANGES = [
    ("2015-01-01", "2015-01-31"),
    ("2016-01-01", "2016-03-31"),
]

print(f"Weather station: Central Park ({NYC_WEATHER_LAT}, {NYC_WEATHER_LON})")
print(f"Target table:    {WEATHER_TABLE}")
print(f"Date ranges:     {DATE_RANGES}")

In [0]:
# ── Fetch hourly weather from Open-Meteo Archive ─────────────────────────────
all_rows = []

for start_date, end_date in DATE_RANGES:
    print(f"Fetching {start_date} to {end_date}...")
    resp = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": NYC_WEATHER_LAT,
            "longitude": NYC_WEATHER_LON,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": "temperature_2m,precipitation,snowfall,wind_speed_10m,cloud_cover",
            "temperature_unit": "fahrenheit",
            "wind_speed_unit": "mph",
            "precipitation_unit": "inch",
            "timezone": "America/New_York",
        },
        timeout=30,
    )
    resp.raise_for_status()
    hourly = resp.json()["hourly"]

    for i, ts in enumerate(hourly["time"]):
        # ts format: "2015-01-01T00:00"
        date_str = ts[:10]
        hour = int(ts[11:13])
        all_rows.append(
            {
                "date": date_str,
                "hour": hour,
                "temperature_f": hourly["temperature_2m"][i],
                "precipitation_inch": hourly["precipitation"][i],
                "snowfall_inch": hourly["snowfall"][i],
                "wind_speed_mph": hourly["wind_speed_10m"][i],
                "cloud_cover_pct": hourly["cloud_cover"][i],
            }
        )
    print(f"  Got {len(hourly['time'])} hourly records")

print(f"\nTotal weather rows: {len(all_rows)}")

In [0]:
# ── Convert to Spark DataFrame and save as Delta ─────────────────────────────
schema = StructType(
    [
        StructField("date", StringType(), False),
        StructField("hour", IntegerType(), False),
        StructField("temperature_f", DoubleType(), True),
        StructField("precipitation_inch", DoubleType(), True),
        StructField("snowfall_inch", DoubleType(), True),
        StructField("wind_speed_mph", DoubleType(), True),
        StructField("cloud_cover_pct", DoubleType(), True),
    ]
)

weather_pdf = pd.DataFrame(all_rows)
weather_df = spark.createDataFrame(weather_pdf, schema=schema)

weather_df.write.format("delta").mode("overwrite").saveAsTable(WEATHER_TABLE)

saved = spark.read.table(WEATHER_TABLE)
print(f"Saved to {WEATHER_TABLE}: {saved.count()} rows")
print(f"Date range: {saved.selectExpr('min(date)', 'max(date)').first()}")
display(saved.orderBy("date", "hour").limit(10))